In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from docx import Document

# Ensure reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Dataset directories
train_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Train'
val_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Val'

# Data augmentation
img_height, img_width = 224, 224
batch_size = 32

datagen_train = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, rotation_range=20, zoom_range=0.2, width_shift_range=0.2, height_shift_range=0.2
)
datagen_val = ImageDataGenerator(rescale=1./255)

train_data = datagen_train.flow_from_directory(
    train_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary'
)
val_data = datagen_val.flow_from_directory(
    val_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary', shuffle=False
)

# Function to save and display confusion matrix
def save_confusion_matrix(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix for {model_name}:\n{cm}")

    results_dir = 'D:\\PROJECT\\PROGRAM\\RESULTS\\'
    os.makedirs(results_dir, exist_ok=True)

    doc = Document()
    doc.add_heading(f'Confusion Matrix for {model_name}', level=1)
    for row in cm:
        doc.add_paragraph(' '.join(map(str, row)))
    doc.save(os.path.join(results_dir, f'{model_name}_confusion_matrix.docx'))

# Function to train and evaluate the CNN model
def train_and_evaluate(model, model_name):
    checkpoint_path = f'D:\\PROJECT\\PROGRAM\\MODELS\\{model_name}.keras'
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max')
    
    print(f"\nTraining {model_name}...\n")
    history = model.fit(train_data, epochs=20, validation_data=val_data, callbacks=[checkpoint])

    best_model = tf.keras.models.load_model(checkpoint_path)
    val_loss, val_accuracy = best_model.evaluate(val_data)
    print(f"{model_name} Validation Accuracy: {val_accuracy * 100:.2f}%")

    y_true = val_data.classes
    y_pred = (best_model.predict(val_data) > 0.5).astype('int32').flatten()

    save_confusion_matrix(model_name, y_true, y_pred)
    return val_accuracy

# CNN Model
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
cnn_accuracy = train_and_evaluate(cnn_model, 'CNN')

# Print CNN Accuracy
print("\nModel Accuracy:")
print(f"CNN Accuracy: {cnn_accuracy * 100:.2f}%")
